# FGTV emissions charts (smoothed with HP filter)

Two charts from `decomposed_ssp_output.csv`, both filtered with the same Hodrick-Prescott smoother used in the R post-processing pipeline (anchored to the first observed year, non-negative).

1. **FGTV emissions by component** — flaring + venting + fugitive (dtp) per strategy.
2. **FGTV emissions by gas** — CO2 + CH4 + N2O per strategy, replicating the CCD "emissions by gas" stack.

Strategy mapping:
- `6003` BAU
- `6004` Unconditional NDC (ZRF)
- `6005` Conditional NDC


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.filters.hp_filter import hpfilter

# Latest run by default (override RUN_DIR below to pin a specific run)
RUNS_ROOT = Path('/Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output')
RUN_DIR = sorted(RUNS_ROOT.glob('sisepuede_results_sisepuede_run_*'))[-1]
print('Run dir:', RUN_DIR.name)

df = pd.read_csv(RUN_DIR / 'decomposed_ssp_output.csv')
att = pd.read_csv(RUN_DIR / 'ATTRIBUTE_PRIMARY.csv')
df = df.merge(att[['primary_id','strategy_id']], on='primary_id')
df['year'] = df['time_period'] + 2015
df = df[df.strategy_id != 0].copy()   # drop BASE
print('strategies:', sorted(df.strategy_id.unique()), '| years:', df.year.min(), '->', df.year.max())

STRATEGY_NAMES = {6003: 'BAU', 6004: 'Unconditional NDC (ZRF)', 6005: 'Conditional NDC'}
STRATEGY_ORDER = [6003, 6004, 6005]
LAMBDA_HP = 1600   # matches R post-processing default for Energy CH4/CO2


## HP filter helper

Hodrick-Prescott smoother, ported from `output_postprocessing/scr/data_prep_new_mapping.r::hp_filter_subsec`:

1. Apply `hpfilter` with `freq=lambda` to the per-strategy series.
2. Floor the trend at 0 (non-negative).
3. Shift so the smoothed series passes exactly through the first observed value (anchor).
4. Floor again, then re-pin the first point.


In [ ]:
def hp_smooth(series: pd.Series, lambda_hp: float = LAMBDA_HP) -> pd.Series:
    """Anchored, non-negative HP trend matching the R hp_filter_subsec output."""
    s = series.astype(float).sort_index()
    if len(s) < 2:
        return s.copy()
    _, trend = hpfilter(s.values, lamb=lambda_hp)
    sm = np.maximum(trend, 0.0)
    shift = s.values[0] - sm[0]
    sm = np.maximum(sm + shift, 0.0)
    sm[0] = s.values[0]
    return pd.Series(sm, index=s.index, name=s.name)


def smooth_by_strategy(df_long: pd.DataFrame, value_cols, lambda_hp: float = LAMBDA_HP) -> pd.DataFrame:
    """Apply hp_smooth to each value_col within each strategy_id, indexed by year."""
    out = df_long.copy()
    for col in value_cols:
        for sid, sub in out.groupby('strategy_id'):
            s = sub.set_index('year')[col]
            sm = hp_smooth(s, lambda_hp = lambda_hp)
            out.loc[sub.index, col] = out.loc[sub.index, 'year'].map(sm)
    return out


## 1 — FGTV emissions by component (flaring + venting + dtp)


In [ ]:
fgtv_cols = [c for c in df.columns if c.startswith('emission_co2e_') and '_fgtv_' in c and c != 'emission_co2e_subsector_total_fgtv']
flar_cols = [c for c in fgtv_cols if '_fgtv_flaring_' in c]
vent_cols = [c for c in fgtv_cols if '_fgtv_venting_' in c]
dtp_cols  = [c for c in fgtv_cols if '_fgtv_dtp_' in c]
print(f'flaring cols: {len(flar_cols)} | venting: {len(vent_cols)} | dtp: {len(dtp_cols)}')

comp = df[['strategy_id','year']].copy()
comp['flaring'] = df[flar_cols].sum(axis=1)
comp['venting'] = df[vent_cols].sum(axis=1)
comp['dtp']     = df[dtp_cols].sum(axis=1)
comp['total']   = comp[['flaring','venting','dtp']].sum(axis=1)

comp_sm = smooth_by_strategy(comp, ['flaring','venting','dtp','total'], lambda_hp = LAMBDA_HP)
comp_sm.head()


In [ ]:
comp_colors = {'flaring': '#f5b04a', 'venting': '#e36d3a', 'dtp': '#7d7d7d'}
strategies_all = [s for s in STRATEGY_ORDER if s in comp_sm.strategy_id.unique()]
fig, axes = plt.subplots(1, len(strategies_all), figsize=(5.0*len(strategies_all), 4.5), sharey=True)
if len(strategies_all) == 1: axes = [axes]
for ax, sid in zip(axes, strategies_all):
    sub = comp_sm[comp_sm.strategy_id == sid].set_index('year').sort_index()
    ax.stackplot(sub.index, sub['flaring'], sub['venting'], sub['dtp'],
                 labels=['Flaring','Venting','Fugitive (dtp)'],
                 colors=[comp_colors['flaring'], comp_colors['venting'], comp_colors['dtp']], alpha=0.85)
    ax.plot(sub.index, sub['total'], 'k--', lw=1.2, label='Total fgtv (subsector)')
    ax.set_title(f"{STRATEGY_NAMES.get(sid, sid)} (id={sid})")
    ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2e'); ax.grid(alpha=0.3)
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
fig.suptitle(f'FGTV emissions by component (HP filter, λ={LAMBDA_HP})')
plt.tight_layout(); plt.show()


## 2 — FGTV emissions by gas (CO2 + CH4 + N2O)

Same FGTV totals, but split by greenhouse gas (in CO2e). Replicates the CCD "emissions by gas" stacked area chart, scoped to FGTV only.


In [ ]:
co2_cols = [c for c in fgtv_cols if c.startswith('emission_co2e_co2_fgtv_')]
ch4_cols = [c for c in fgtv_cols if c.startswith('emission_co2e_ch4_fgtv_')]
n2o_cols = [c for c in fgtv_cols if c.startswith('emission_co2e_n2o_fgtv_')]
print(f'CO2 cols: {len(co2_cols)} | CH4 cols: {len(ch4_cols)} | N2O cols: {len(n2o_cols)}')

gas = df[['strategy_id','year']].copy()
gas['CO2'] = df[co2_cols].sum(axis=1)
gas['CH4'] = df[ch4_cols].sum(axis=1)
gas['N2O + F-gases'] = df[n2o_cols].sum(axis=1)   # FGTV has no F-gases; column kept for visual parity with CCD chart
gas['total'] = gas[['CO2','CH4','N2O + F-gases']].sum(axis=1)

gas_sm = smooth_by_strategy(gas, ['CO2','CH4','N2O + F-gases','total'], lambda_hp = LAMBDA_HP)
gas_sm.head()


In [ ]:
gas_colors = {'CO2': '#9bb7d4', 'CH4': '#f4a14a', 'N2O + F-gases': '#f3c1c4'}
fig, axes = plt.subplots(1, len(strategies_all), figsize=(5.0*len(strategies_all), 4.5), sharey=True)
if len(strategies_all) == 1: axes = [axes]
anchor_year = 2035
for ax, sid in zip(axes, strategies_all):
    sub = gas_sm[gas_sm.strategy_id == sid].set_index('year').sort_index()
    ax.stackplot(sub.index, sub['CO2'], sub['CH4'], sub['N2O + F-gases'],
                 labels=['CO2','CH4','N2O + F-gases'],
                 colors=[gas_colors['CO2'], gas_colors['CH4'], gas_colors['N2O + F-gases']], alpha=0.9)
    ax.plot(sub.index, sub['total'], 'k-', lw=1.4)
    if anchor_year in sub.index:
        ax.axvline(anchor_year, color='gray', ls='--', lw=1, alpha=0.6)
        ax.annotate(f"{sub.loc[anchor_year,'total']:.2f}", xy=(anchor_year, sub.loc[anchor_year,'total']),
                    xytext=(4, 4), textcoords='offset points', fontsize=9)
    last_year = sub.index.max()
    ax.annotate(f"{sub.loc[last_year,'total']:.2f}", xy=(last_year, sub.loc[last_year,'total']),
                xytext=(4, 4), textcoords='offset points', fontsize=9)
    ax.set_title(f"{STRATEGY_NAMES.get(sid, sid)} (id={sid})")
    ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2e'); ax.grid(alpha=0.3)
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, title='Gas (group)')
fig.suptitle(f'FGTV emissions by gas (HP filter, λ={LAMBDA_HP})')
plt.tight_layout(); plt.show()


## 3 — FGTV volumes (Bcm, MMcfd) and gas recovery

Convert the FGTV emissions to natural-gas-equivalent volumes using GGFR / IEA conventions and calibrate to the Libya CCD March 2026 anchors (2024 BAU: flaring 6.30 Bcm, venting 0.98 Bcm, fugitive 0.33 Bcm).

`gas_recovery` is defined as the **avoided CO2e emissions** vs the BAU twin, expressed as flared-equivalent Bcm (so the unit stays consistent with the other volume bands):

```
gas_recovery_mtco2e = BAU(flaring + venting + fugitive)_mtco2e - scenario(flaring + venting + fugitive)_mtco2e
gas_recovery_bcm    = gas_recovery_mtco2e / 2.54  # 1 Mt CO2 ≈ 0.394 Bcm of flared gas
```

We weight by emissions instead of physical volume because the three FGTV streams have very different climate impact per m³ (vented CH4 carries ~7× more CO2e than flared CO2). A scenario that pushes gas from flaring into venting cuts physical volume but not emissions; under the volume-only formula it would falsely report large recovery, while this emissions-weighted formula tracks the actual climate gain.

`gas_recovery` is zero for BAU by construction.


In [ ]:
# Physical conversion constants (15 degC, 1 atm)
KG_CO2_PER_M3_FLARED_GAS = 2.54   # composition-weighted CO2 yield from associated gas combustion
KG_CH4_PER_M3_PURE       = 0.717  # density of pure CH4 at NTP
NG_CH4_VOLUME_FRACTION   = 0.95   # vented/leaked stream ~95% CH4 by volume
GWP_CH4                  = 27.9   # IPCC AR6 100-year, matches sisepuede attribute_gas
MMCFD_PER_BCM            = 96.75  # 1 Bcm/year continuous flow ≈ 96.75 MMcfd

# 2024 BAU report anchors (Libya CCD, World Bank, March 2026)
TARGET_2024_BAU_BCM = {'flaring': 6.30, 'venting': 0.981, 'fugitive': 0.334}

def co2_emissions_mt_to_volume_bcm(em_mt):
    """Flaring: tonnes CO2 -> Bcm of associated gas combusted."""
    return em_mt * 1e9 / KG_CO2_PER_M3_FLARED_GAS / 1e9

def ch4_emissions_co2eq_mt_to_volume_bcm(em_mt):
    """Venting/fugitive: Mt CO2eq of CH4 -> Bcm of natural gas escaped."""
    return em_mt * 1e9 / GWP_CH4 / KG_CH4_PER_M3_PURE / NG_CH4_VOLUME_FRACTION / 1e9

# Use CO2 emissions for flaring (combustion product), CH4 for venting/fugitive (escape products).
flar_co2_cols = [c for c in df.columns if c.startswith('emission_co2e_co2_fgtv_flaring_')]
vent_ch4_cols = [c for c in df.columns if c.startswith('emission_co2e_ch4_fgtv_venting_')]
dtp_ch4_cols  = [c for c in df.columns if c.startswith('emission_co2e_ch4_fgtv_dtp_')]

vol = df[['strategy_id','year']].copy()
vol['flaring_bcm']  = co2_emissions_mt_to_volume_bcm(df[flar_co2_cols].sum(axis=1))
vol['venting_bcm']  = ch4_emissions_co2eq_mt_to_volume_bcm(df[vent_ch4_cols].sum(axis=1))
vol['fugitive_bcm'] = ch4_emissions_co2eq_mt_to_volume_bcm(df[dtp_ch4_cols].sum(axis=1))

# Production volumes (oil and gas) from PJ + density.
for fuel in ['crude','natural_gas']:
    prod_col = f'prod_enfu_fuel_{fuel}_pj'
    dens_col = f'energydensity_enfu_mj_per_litre_fuel_{fuel}'
    if prod_col in df.columns and dens_col in df.columns:
        density = df[dens_col].replace(0, np.nan)
        vol[f'production_{fuel}_bcm'] = (df[prod_col] * 1e6 / density / 1e9).fillna(0.0)

# HP-smooth raw volumes per strategy. We calibrate AFTER smoothing so the
# 2024 BAU smoothed value matches the report exactly (HP anchors at 2023).
vol_sm = smooth_by_strategy(
    vol,
    ['flaring_bcm','venting_bcm','fugitive_bcm','production_crude_bcm','production_natural_gas_bcm'],
    lambda_hp = LAMBDA_HP,
)

# Calibrate each FGTV pathway to the report's 2024 BAU anchor.
calibration = {}
for path in ['flaring','venting','fugitive']:
    smoothed_2024_bau = vol_sm.loc[
        (vol_sm.strategy_id == 6003) & (vol_sm.year == 2024), f'{path}_bcm'
    ].iloc[0]
    factor = TARGET_2024_BAU_BCM[path] / smoothed_2024_bau if smoothed_2024_bau > 0 else 1.0
    calibration[path] = factor
    vol_sm[f'{path}_bcm'] *= factor
print('Calibration factors (smoothed 2024 BAU -> CCD 2024 target):',
      {k: round(v, 3) for k, v in calibration.items()})

# Implied CO2e emissions per pathway from the calibrated volumes. We need
# these to compute gas_recovery in emissions-weighted terms (CH4 venting
# carries ~7x more CO2e per Bcm than CO2 flaring, so a pure-volume diff
# under-counts the climate value of avoiding venting).
vol_sm['flaring_mtco2e']  = vol_sm['flaring_bcm']  * 1e9 * KG_CO2_PER_M3_FLARED_GAS / 1e9
vol_sm['venting_mtco2e']  = vol_sm['venting_bcm']  * 1e9 * KG_CH4_PER_M3_PURE * NG_CH4_VOLUME_FRACTION * GWP_CH4 / 1e9
vol_sm['fugitive_mtco2e'] = vol_sm['fugitive_bcm'] * 1e9 * KG_CH4_PER_M3_PURE * NG_CH4_VOLUME_FRACTION * GWP_CH4 / 1e9
vol_sm['fgtv_total_mtco2e'] = vol_sm[['flaring_mtco2e','venting_mtco2e','fugitive_mtco2e']].sum(axis=1)

# gas_recovery measured as avoided CO2e emissions vs the BAU twin, then
# expressed as flared-equivalent Bcm so the chart unit stays consistent.
# Definition: 'how many Bcm of associated gas would have had to NOT be
# flared in BAU to produce the avoided MtCO2e seen in this scenario.'
bau_total_co2e_by_year = vol_sm.loc[vol_sm.strategy_id == 6003].set_index('year')['fgtv_total_mtco2e']
vol_sm['fgtv_total_mtco2e_bau'] = vol_sm['year'].map(bau_total_co2e_by_year)
vol_sm['gas_recovery_mtco2e']   = vol_sm['fgtv_total_mtco2e_bau'] - vol_sm['fgtv_total_mtco2e']
vol_sm['gas_recovery_bcm']      = vol_sm['gas_recovery_mtco2e'] / KG_CO2_PER_M3_FLARED_GAS

# MMcfd convenience columns
for c in ['flaring','venting','fugitive','gas_recovery']:
    vol_sm[f'{c}_mmcfd'] = vol_sm[f'{c}_bcm'] * MMCFD_PER_BCM

vol_sm.head()


In [ ]:
# Sanity check: 2024 BAU should match the report; gas_recovery should be 0 for BAU
# and reflect avoided MtCO2e for the scenarios.
sanity = vol_sm[(vol_sm.year.isin([2024, 2030, 2035, 2050])) & (vol_sm.strategy_id.isin(STRATEGY_ORDER))][[
    'year','strategy_id',
    'flaring_bcm','venting_bcm','fugitive_bcm',
    'fgtv_total_mtco2e','gas_recovery_mtco2e','gas_recovery_bcm',
]].copy()
sanity['strategy'] = sanity['strategy_id'].map(STRATEGY_NAMES)
sanity = sanity.round(3)
print(sanity.sort_values(['year','strategy_id']).drop(columns='strategy_id').to_string(index=False))

print('\nLibya CCD March 2026 anchors:')
print('  2024 BAU flaring  6.30 Bcm | venting 0.98 Bcm | fugitive 0.33 Bcm')
print('  2030 BAU flaring ~10.25 Bcm (~991 MMcfd)')
print('  2030 ZRF flaring  0.94 Bcm (~91 MMcfd)')


## Export for Tableau

Long-format CSV with the same schema used by the existing Tableau workbook: `year, strategy_id, strategy_name, chart, metric, facility_type, value, units`. Smoothed values are exported (HP filter applied).

- `chart='fgtv_components'`: metrics `flar_emis_mt_co2e`, `vent_ch4_mt_co2e`, `dtp_ch4_mt_co2e`, `total_fgtv_mt_co2e`.
- `chart='fgtv_by_gas'`: metrics `fgtv_co2_mt_co2e`, `fgtv_ch4_mt_co2e`, `fgtv_n2o_fgases_mt_co2e`, `total_fgtv_mt_co2e`.
- `chart='fgtv_volumes'`: metrics `flaring_vol_bcm`, `venting_vol_bcm`, `fugitive_vol_bcm`, `production_oil_vol_bcm`, `production_gas_vol_bcm`, plus `*_vol_mmcfd` variants, plus `gas_recovery_mt_co2e` (avoided CO2e vs BAU) and `gas_recovery_vol_bcm` / `gas_recovery_vol_mmcfd` (the same value expressed as flared-equivalent gas volume).

Saved to `ssp_modeling/tableau/data/oilgas_ccd_charts_libya.csv` (overwriting prior file).


In [ ]:
rows = []

# Chart A: FGTV components — emissions (Mt CO2e), smoothed
comp_metric_map = {
    'flaring': ('flar_emis_mt_co2e', 'Mt CO2e'),
    'venting': ('vent_ch4_mt_co2e',  'Mt CO2e'),
    'dtp':     ('dtp_ch4_mt_co2e',   'Mt CO2e'),
    'total':   ('total_fgtv_mt_co2e','Mt CO2e'),
}
for _, r in comp_sm.iterrows():
    sid = int(r['strategy_id'])
    for col, (metric, units) in comp_metric_map.items():
        rows.append({
            'year': int(r['year']),
            'strategy_id': sid,
            'strategy_name': STRATEGY_NAMES.get(sid, str(sid)),
            'chart': 'fgtv_components',
            'metric': metric,
            'facility_type': 'All',
            'value': float(r[col]),
            'units': units,
        })

# Chart B: FGTV by gas — emissions (Mt CO2e), smoothed
gas_metric_map = {
    'CO2':            ('fgtv_co2_mt_co2e',         'Mt CO2e'),
    'CH4':            ('fgtv_ch4_mt_co2e',         'Mt CO2e'),
    'N2O + F-gases':  ('fgtv_n2o_fgases_mt_co2e',  'Mt CO2e'),
    'total':          ('total_fgtv_mt_co2e',       'Mt CO2e'),
}
for _, r in gas_sm.iterrows():
    sid = int(r['strategy_id'])
    for col, (metric, units) in gas_metric_map.items():
        rows.append({
            'year': int(r['year']),
            'strategy_id': sid,
            'strategy_name': STRATEGY_NAMES.get(sid, str(sid)),
            'chart': 'fgtv_by_gas',
            'metric': metric,
            'facility_type': 'All',
            'value': float(r[col]),
            'units': units,
        })

# Chart C: FGTV volumes (Bcm + MMcfd) and gas_recovery (MtCO2e + Bcm)
vol_metric_map = {
    'flaring_bcm':              ('flaring_vol_bcm',           'Bcm'),
    'venting_bcm':              ('venting_vol_bcm',           'Bcm'),
    'fugitive_bcm':             ('fugitive_vol_bcm',          'Bcm'),
    'production_crude_bcm':     ('production_oil_vol_bcm',    'Bcm'),
    'production_natural_gas_bcm':('production_gas_vol_bcm',   'Bcm'),
    'flaring_mmcfd':            ('flaring_vol_mmcfd',         'MMcfd'),
    'venting_mmcfd':            ('venting_vol_mmcfd',         'MMcfd'),
    'fugitive_mmcfd':           ('fugitive_vol_mmcfd',        'MMcfd'),
    # gas_recovery is emissions-weighted (avoided CO2e vs BAU twin),
    # exported in both MtCO2e (climate metric) and flared-equivalent Bcm/MMcfd.
    'gas_recovery_mtco2e':      ('gas_recovery_mt_co2e',      'Mt CO2e'),
    'gas_recovery_bcm':         ('gas_recovery_vol_bcm',      'Bcm'),
    'gas_recovery_mmcfd':       ('gas_recovery_vol_mmcfd',    'MMcfd'),
}
for _, r in vol_sm.iterrows():
    sid = int(r['strategy_id'])
    for col, (metric, units) in vol_metric_map.items():
        if col not in vol_sm.columns:
            continue
        rows.append({
            'year': int(r['year']),
            'strategy_id': sid,
            'strategy_name': STRATEGY_NAMES.get(sid, str(sid)),
            'chart': 'fgtv_volumes',
            'metric': metric,
            'facility_type': 'All',
            'value': float(r[col]),
            'units': units,
        })

tab = pd.DataFrame(rows, columns=['year','strategy_id','strategy_name','chart','metric','facility_type','value','units'])
out_path = Path('/Users/fabianfuentes/git/ssp_libya/ssp_modeling/tableau/data/oilgas_ccd_charts_libya.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
tab.to_csv(out_path, index=False)
print(f'Wrote {len(tab)} rows to {out_path}')
print('chart values :', sorted(tab.chart.unique()))
print('metric values:', sorted(tab.metric.unique()))
tab.head()


## Notes

- `LAMBDA_HP=1600` matches the R post-processing default used for Energy CH4/CO2 series. Use 600 for noisier series, 100 for almost-no-smoothing.
- The first observed year (2023) is anchored exactly to the model output. Smoothing only affects later years.
- The dashed black line on chart 1 is the FGTV subsector total (sanity check: should equal the stacked sum).
